In [2]:
query = """
SELECT
  p.product_category_name_english,
  SUM(oi.price) AS total_sales_value
FROM `olist-data-pipeline-507001.olist_mart.fact_order_items` oi
JOIN `olist-data-pipeline-507001.olist_mart.fact_orders` o
  ON oi.order_key = o.order_key
JOIN `olist-data-pipeline-507001.olist_mart.dim_product` p
  ON oi.product_id = p.product_id
WHERE o.order_status = 'DELIVERED'
GROUP BY p.product_category_name_english
ORDER BY total_sales_value DESC
"""

category_sales = pd.read_sql(query, engine)

category_sales.head(10)

/home/nivedha/miniconda3/lib/python3.14/site-packages/google/cloud/bigquery/client.py:623: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


,product_category_name_english,total_sales_value
0,health_beauty,1224543.61
1,watches_gifts,1159219.92
2,bed_bath_table,1013590.72
3,sports_leisure,948960.45
4,computers_accessories,884760.96
5,furniture_decor,705997.44
6,housewares,611635.68
7,cool_stuff,603221.05
8,auto,571453.35
9,garden_tools,467301.19


In [3]:
query = """
SELECT
  p.product_category_name_english,
  SUM(oi.price) AS total_sales_value,
  SAFE_DIVIDE(
    SUM(oi.price),
    SUM(SUM(oi.price)) OVER ()
  ) * 100 AS sales_share_percent
FROM `olist-data-pipeline-507001.olist_mart.fact_order_items` oi
JOIN `olist-data-pipeline-507001.olist_mart.fact_orders` o
  ON oi.order_key = o.order_key
JOIN `olist-data-pipeline-507001.olist_mart.dim_product` p
  ON oi.product_id = p.product_id
WHERE o.order_status = 'DELIVERED'
GROUP BY p.product_category_name_english
ORDER BY total_sales_value DESC
"""

category_sales = pd.read_sql(query, engine)

category_sales.head(10)

,product_category_name_english,total_sales_value,sales_share_percent
0,health_beauty,1224543.61,9.340632
1,watches_gifts,1159219.92,8.842353
2,bed_bath_table,1013590.72,7.731515
3,sports_leisure,948960.45,7.238525
4,computers_accessories,884760.96,6.748822
5,furniture_decor,705997.44,5.385241
6,housewares,611635.68,4.665464
7,cool_stuff,603221.05,4.601278
8,auto,571453.35,4.358959
9,garden_tools,467301.19,3.564502


In [5]:
# Calculate cumulative sales share

category_sales = category_sales.sort_values(
    "total_sales_value",
    ascending=False
).reset_index(drop=True)

category_sales["cumulative_sales_value"] = (
    category_sales["total_sales_value"].cumsum()
)

category_sales["cumulative_sales_share_percent"] = (
    category_sales["cumulative_sales_value"]
    / category_sales["total_sales_value"].sum()
    * 100
)

category_sales.head(10)

,product_category_name_english,total_sales_value,sales_share_percent,cumulative_sales_value,cumulative_sales_share_percent
0,health_beauty,1224543.61,9.340632,1224543.61,9.340632
1,watches_gifts,1159219.92,8.842353,2383763.53,18.182984
2,bed_bath_table,1013590.72,7.731515,3397354.25,25.914499
3,sports_leisure,948960.45,7.238525,4346314.70,33.153025
4,computers_accessories,884760.96,6.748822,5231075.66,39.901846
5,furniture_decor,705997.44,5.385241,5937073.10,45.287087
6,housewares,611635.68,4.665464,6548708.78,49.952551
7,cool_stuff,603221.05,4.601278,7151929.83,54.553829
8,auto,571453.35,4.358959,7723383.18,58.912788
9,garden_tools,467301.19,3.564502,8190684.37,62.477290


In [6]:
# Top 10 individual products by delivered-order sales value

query = """
SELECT
  p.product_id,
  p.product_category_name_english,
  SUM(oi.price) AS total_sales_value
FROM `olist-data-pipeline-507001.olist_mart.fact_order_items` oi
JOIN `olist-data-pipeline-507001.olist_mart.fact_orders` o
  ON oi.order_key = o.order_key
JOIN `olist-data-pipeline-507001.olist_mart.dim_product` p
  ON oi.product_id = p.product_id
WHERE o.order_status = 'DELIVERED'
GROUP BY
  p.product_id,
  p.product_category_name_english
ORDER BY total_sales_value DESC
LIMIT 10
"""

top_products = pd.read_sql(query, engine)

top_products

,product_id,product_category_name_english,total_sales_value
0,bb50f2e236e5eea0100680137654686c,health_beauty,63560.00
1,6cdd53843498f92890544667809f1595,health_beauty,53302.40
2,d6160fb7873f184099d9bc95e30376af,computers,45949.35
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,45322.56
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,41330.46
5,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,40632.90
6,25c38557cf793876c5abdd5931f922db,baby,37908.32
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,37733.90
8,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,37212.84
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,36884.40


In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("bigquery://olist-data-pipeline-507001")